# Part 3 : Snowflake CoWork

## このノートブックでやること
1. **データ確認** — バナー広告クリエイティブと配信実績データを確認する
2. **バナー画像の AI 分析** — 画像から5つのクリエイティブ特徴を自動抽出する
3. **Semantic View の作成** — データの意味と関係を定義する
4. **Snowflake CoWork で質問してみよう** — 成功例を体験する
5. **CoWork が答えられない質問をしてみよう** — 失敗例から原因を理解する
6. **Semantic View を改善する** — 効果スコアのメトリクスを追加する
7. **再チャレンジ！** — 改善後に同じ質問をして違いを確認する

## シナリオ
金融系 Web バナー広告の **A/B テストデータ** を使って、「どんなクリエイティブが高い効果スコアを上げるか？」を分析します。

| テーブル | 内容 |
|---|---|
| `raw_ad_creatives` | 広告配信実績（インプレッション・クリック・コンバージョンなど） |
| `gold_ad_image_analysis` | AI が画像から抽出したクリエイティブ特徴量（5項目） |

In [ ]:
%%sql -r dataframe_3_0
-- 環境設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE GLACIERSTYLE_WH;
USE DATABASE GLACIERSTYLE_DB;
USE SCHEMA EC_ANALYTICS_SCHEMA;

## 1. データ確認

バナー広告データ（`raw_ad_creatives`）を確認します。

このテーブルには以下の情報が含まれています：
- **クリエイティブ情報**: 広告名、商品ジャンル（投資 / 保険 / クレジットカード）
- **配信実績**: インプレッション数・クリック数・コンバージョン数・広告費用
- **画像参照**: バナー画像ファイル名（`image_file_path`）

金融バナー広告データ（`FIN-001`〜`FIN-008`）は `setup.sql` 実行時に投入済みです。

今回 SI で分析に使うデータを確認しておきましょう。

### 使用するテーブル

| テーブル | 内容 |
|---|---|
| `raw_sns_mentions` | インフルエンサーの SNS 投稿（いいね・RT 数など） |
| `fact_orders` | 注文データ（売上金額・日時など） |
| `dim_products` | 商品マスタ（商品名・カテゴリなど） |

In [ ]:
%%sql -r dataframe_3_1
-- 投入したデータを確認
SELECT
    creative_id,
    creative_name,
    image_file_path,
    target_segment,
    impressions,
    clicks,
    conversions,
    ROUND(spend)                                            AS spend,
    ROUND(DIV0(clicks, impressions) * 100, 2)             AS ctr_pct,
    ROUND(DIV0(conversions, clicks) * 100, 2)             AS cvr_pct
FROM raw_ad_creatives
WHERE creative_id LIKE 'FIN-%'
ORDER BY creative_id;

## 2. バナー広告画像の AI 分析

`AI_COMPLETE` を使って、バナー画像から5つのクリエイティブ特徴量を自動抽出します。

| 特徴量 | 型 | 説明 |
|---|---|---|
| `has_person` | BOOLEAN | 人物（顔・体・手など）が写っているか |
| `has_mascot` | BOOLEAN | マスコットキャラクターが登場するか |
| `color_tone` | VARCHAR | 色調（暖色系 / 寒色系） |
| `is_complex` | BOOLEAN | 情報量が多い複雑な構成か |
| `has_number_highlight` | BOOLEAN | 数字（%・金利・円）が強調されているか |

抽出した特徴量は `gold_ad_image_analysis` テーブルに保存します。

SNS の投稿には**画像**が添付されています。テキストデータだけでなく、画像の特徴もバズスコアに影響するかもしれません。

Snowflake の AI 関数を使って、投稿画像から特徴を自動抽出します。

### 抽出する特徴

| 特徴 | 内容 |
|---|---|
| 人物の有無 | 人が写っているか |
| 表情 | ポジティブ / ネガティブ / ニュートラル |
| 画像の明るさ | 明るい / 暗い |
| 商品の視認性 | 商品がはっきり映っているか |

In [ ]:
%%sql -r dataframe_3_2
-- ステージ上のバナー画像ファイルを確認（64件あること）
SELECT
    relative_path,
    size,
    last_modified
FROM DIRECTORY(@DATA_STAGE)
WHERE relative_path LIKE 'data/images/part3/%_%.png'
ORDER BY relative_path;

In [ ]:
%%sql -r dataframe_3_3
-- 1枚の画像で AI 分析のテスト（001_1.png）
-- JSON が正しく返ってくるか確認する
SELECT
    relative_path,
    PARSE_JSON(
        REGEXP_REPLACE(
            AI_COMPLETE(
                'claude-sonnet-4-6',
                PROMPT(
                    $$以下の金融広告バナー画像を分析して、JSON形式で結果を返してください。
JSONオブジェクトのみ返してください。説明文やコードブロック記法は不要です。

has_person: true or false
  -- 広告バナーに人物（顔・体・手など身体の一部）が写っているか
has_mascot: true or false
  -- 広告バナーにマスコットキャラクター（動物・キャラクター・ゆるキャラなど）が登場するか。実在の人物は含まない
color_tone: "暖色系" or "寒色系"
  -- 画像全体の色調。赤・オレンジ・黄が主体なら暖色系、青・紺・緑が主体なら寒色系
is_complex: true or false
  -- グラフ・表・複数のテキストブロックなど情報要素が多く複雑な構成か。テキストと要素が少なくシンプルならfalse
has_number_highlight: true or false
  -- 利率（%）・金額（円）・期間・還元率など具体的な数字が大きく・目立つ形で強調されているか

画像: {0}$$,
                    TO_FILE('@DATA_STAGE', relative_path)
                )
            ),
            '^[^{]*|[^}]*$', ''
        )
    ) AS features_json
FROM DIRECTORY(@DATA_STAGE)
WHERE relative_path = 'data/images/part3/001_1.png';

In [ ]:
%%sql -r dataframe_3_4
-- 全バナー画像を分析して gold テーブルに保存
-- ※ AI_COMPLETE が全件実行されるので数秒〜数十秒かかります
CREATE OR REPLACE TABLE gold_ad_image_analysis AS
SELECT
    SPLIT_PART(relative_path, '/', -1)                   AS image_file,
    relative_path,
    PARSE_JSON(
        REGEXP_REPLACE(
            AI_COMPLETE(
                'claude-sonnet-4-6',
                PROMPT(
                    $$以下の金融広告バナー画像を分析して、JSON形式で結果を返してください。
JSONオブジェクトのみ返してください。説明文やコードブロック記法は不要です。

has_person: true or false
  -- 広告バナーに人物（顔・体・手など身体の一部）が写っているか
has_mascot: true or false
  -- 広告バナーにマスコットキャラクター（動物・キャラクター・ゆるキャラなど）が登場するか。実在の人物は含まない
color_tone: "暖色系" or "寒色系"
  -- 画像全体の色調。赤・オレンジ・黄が主体なら暖色系、青・紺・緑が主体なら寒色系
is_complex: true or false
  -- グラフ・表・複数のテキストブロックなど情報要素が多く複雑な構成か。テキストと要素が少なくシンプルならfalse
has_number_highlight: true or false
  -- 利率（%）・金額（円）・期間・還元率など具体的な数字が大きく・目立つ形で強調されているか

画像: {0}$$,
                    TO_FILE('@DATA_STAGE', relative_path)
                )
            ),
            '^[^{]*|[^}]*$', ''
        )
    )                                                     AS features_json,
    features_json:has_person::BOOLEAN                    AS has_person,
    features_json:has_mascot::BOOLEAN                    AS has_mascot,
    features_json:color_tone::VARCHAR                    AS color_tone,
    features_json:is_complex::BOOLEAN                    AS is_complex,
    features_json:has_number_highlight::BOOLEAN          AS has_number_highlight
FROM DIRECTORY(@DATA_STAGE)
WHERE relative_path LIKE 'data/images/part3/%_%.png'
ORDER BY relative_path;

-- 結果確認
SELECT * FROM gold_ad_image_analysis ORDER BY image_file;

## 3. Semantic View を作成する

**Semantic View（セマンティックビュー）** は、CoWork が「このデータは何を意味するか」を理解するためのメタデータ定義です。

- テーブルの意味や関係性を定義する
- ビジネス指標（メトリクス）をあらかじめ定義しておく
- CoWork はこの定義をもとに自然言語クエリを SQL に変換する

### ポイント

Semantic View に **定義されていないメトリクスや結合は CoWork が答えられません。**
後のセクションでこれを体験します。

### 今回作成する Semantic View（Before 版）

| 要素 | 内容 |
|---|---|
| ベーステーブル | `raw_ad_creatives`（広告配信実績） |
| メトリクス | インプレッション・クリック・コンバージョン・CTR |
| ※ 未定義 | 効果スコア、画像分析テーブルとの結合 |

In [ ]:
%%sql -r dataframe_3_5
-- =====================================================
-- Semantic View の作成（Before 版）
-- raw_ad_creatives のみ。ACPスコアと画像分析は未定義。
-- =====================================================
CREATE OR REPLACE SEMANTIC VIEW sv_ad_creative_analysis
  TABLES (
    raw_ad_creatives AS GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.RAW_AD_CREATIVES
      PRIMARY KEY (creative_id)
  )
  DIMENSIONS (
    raw_ad_creatives.creative_id AS creative_id
      COMMENT = '広告クリエイティブID',
    raw_ad_creatives.creative_name AS creative_name
      COMMENT = '広告名',
    raw_ad_creatives.image_file_path AS image_file_path
      COMMENT = 'バナー画像ファイル名',
    raw_ad_creatives.target_segment AS target_segment
      COMMENT = '商品ジャンル（投資 / 保険 / クレジットカード）',
    raw_ad_creatives.platform AS platform
      COMMENT = '配信プラットフォーム',
    raw_ad_creatives.start_date AS start_date
      COMMENT = '配信開始日',
    raw_ad_creatives.end_date AS end_date
      COMMENT = '配信終了日',
    raw_ad_creatives.impressions AS impressions
      COMMENT = 'インプレッション数',
    raw_ad_creatives.clicks AS clicks
      COMMENT = 'クリック数',
    raw_ad_creatives.conversions AS conversions
      COMMENT = 'コンバージョン数',
    raw_ad_creatives.spend AS spend
      COMMENT = '広告費用（円）'
  );

In [ ]:
%%sql -r dataframe_3_6
-- 作成した Semantic View を確認
DESCRIBE SEMANTIC VIEW sv_ad_creative_analysis;

## 4. Snowflake CoWork で質問してみよう

Semantic View の準備ができました。いよいよ CoWork を使ってみましょう。

### CoWork の起動手順

#### Step 1: Cortex Agent を作成する

> **注意:** エージェントの作成には権限が必要です。ロールを **ACCOUNTADMIN** に切り替えてから操作してください。

1. Snowsight の左メニューから **「AI & ML」→「エージェント」** をクリック
2. 右上の **「+ エージェント」** をクリック
3. データベースとスキーマを選択（`GLACIERSTYLE_DB` / `EC_ANALYTICS_SCHEMA`）
4. エージェント名を入力（例: `Ad_Performance_Navi`）
5. **「エージェントを作成」** をクリック

#### Step 2: Semantic View をツールとして追加する

1. 作成したエージェントの設定画面で 「ツール」タブを開き、**「Cortex アナリスト」** の「＋追加」をクリック
2. **「セマンティックビューを追加」** をクリック
3. 名前を入力（例: `ad_performance_analysis`）
4. 説明を入力（例: `広告クリエイティブの配信実績と効果スコアを分析する`）
5. `GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.SV_AD_CREATIVE_ANALYSIS` を選択
6. ウェアハウスに `GLACIERSTYLE_WH` を指定
7. 「保存」をクリック

#### Step 3: CoWork でエージェントに質問する

1. Snowsight の左メニューから **「AI & ML」→「Snowflake Intelligence」** をクリックしてSnowflake Intelligenceを開く
2. 作成したエージェント（`Ad_Performance_Navi`）を選択
3. チャット欄に質問を入力

### 試してみる質問（シンプルな集計）

まずは CoWork が確実に答えられる**シンプルな質問**を試してみましょう。

```
クリック率（CTR）が高い商品ジャンルを教えてください
```

```
コンバージョン数の合計を商品ジャンル別に見せてください
```

> **確認ポイント:** CoWork が SQL を自動生成して結果を返してくれることを確認しましょう。

In [ ]:
%%sql -r dataframe_3_7
-- 参考: CoWork が生成する SQL と同等のクエリ（手動で確認）
-- CoWork の回答と結果が一致するか検証に使えます
SELECT
    target_segment,
    SUM(impressions)                                           AS total_impressions,
    SUM(clicks)                                               AS total_clicks,
    SUM(conversions)                                          AS total_conversions,
    ROUND(DIV0(SUM(clicks), SUM(impressions)) * 100, 2)      AS ctr_pct
FROM raw_ad_creatives
WHERE creative_id LIKE 'FIN-%'
GROUP BY target_segment
ORDER BY ctr_pct DESC;

## 5. CoWork が答えられない質問をしてみよう

今度は少し**踏み込んだ質問**を CoWork にしてみましょう。

### 試してみる質問

```
ACPスコアが高い広告クリエイティブの傾向を教えてください
```

```
人物が写っている広告とそうでない広告で、効果に差はありますか？
```

```
暖色系と寒色系のバナーでクリック率に差はありますか？
```

> **予想:** CoWork はこれらの質問に答えられないか、的外れな回答をするはずです。

### なぜ答えられないのか？

**2つの理由があります。**

1. **「ACPスコア」という概念が Semantic View に定義されていないから**
   - ACPスコア = CTR × CVR × 10,000（社内 KPI）
   - この計算式が SV に登録されていないため、CoWork は意味を理解できない

2. **画像分析テーブル（`gold_ad_image_analysis`）が SV に含まれていないから**
   - `has_person`・`color_tone` などの特徴量が参照されていない
   - `raw_ad_creatives` との結合も定義されていない

→ **Semantic View を改善して、このような質問にも答えられるようにしましょう！**

## 6. Semantic View を改善する（新しい View を作成）

改善前の `sv_ad_creative_analysis` はそのまま残し、**新しい Semantic View `sv_ad_performance_enhanced`** を作成します。

**ACPスコア** のメトリクスと **画像分析テーブル** を追加した強化版です。

### METRICS とは？

Before版では DIMENSIONS（実カラム）だけを定義しましたが、ここで新たに **METRICS** を使います。

| 区分 | 役割 | 例 |
|---|---|---|
| DIMENSIONS | テーブルに実在するカラム（生データ） | `clicks`, `impressions`, `target_segment` |
| **METRICS** | **計算式で導出される指標** | `DIV0(SUM(conversions), SUM(impressions)) * 10000` |

METRICS を定義することで、Analyst は「ACPスコアを出して」と聞かれたときに正しい計算式を使えるようになります。

### ACPスコアの定義

| 指標 | 計算式 | 意味 |
|---|---|---|
| CTR | clicks ÷ impressions | クリック率（表示からクリックされる確率） |
| CVR | conversions ÷ clicks | コンバージョン率（クリックから成約する確率） |
| **ACPスコア** | **CTR × CVR × 10,000** | 1回表示あたりの成約確率を10,000倍した社内 KPI |

ACPスコアが高いほど「1回の広告表示から成約に至る確率が高い」クリエイティブです。

### 改善するポイント

1. `acp_score`（ACPスコア）を **METRICS** として追加
2. `gold_ad_image_analysis`（画像分析テーブル）を SV に追加
3. `raw_ad_creatives` と `gold_ad_image_analysis` の **RELATIONSHIP** を定義

In [ ]:
-- =====================================================
-- Semantic View 改善版を新規作成（After 版）
-- ACPスコア + 画像分析テーブルを追加した新しい View
-- =====================================================
CREATE OR REPLACE SEMANTIC VIEW sv_ad_performance_enhanced
  TABLES (
    raw_ad_creatives AS GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.RAW_AD_CREATIVES
      PRIMARY KEY (creative_id),
    gold_ad_image_analysis AS GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.GOLD_AD_IMAGE_ANALYSIS
      PRIMARY KEY (image_file)
  )
  RELATIONSHIPS (
    creatives_to_images AS
      raw_ad_creatives (image_file_path) REFERENCES gold_ad_image_analysis (image_file)
  )
  DIMENSIONS (
    raw_ad_creatives.creative_id AS creative_id
      COMMENT = '広告クリエイティブID',
    raw_ad_creatives.creative_name AS creative_name
      COMMENT = '広告名',
    raw_ad_creatives.image_file_path AS image_file_path
      COMMENT = 'バナー画像ファイル名（gold_ad_image_analysis との結合キー）',
    raw_ad_creatives.target_segment AS target_segment
      COMMENT = '商品ジャンル（投資 / 保険 / クレジットカード）',
    raw_ad_creatives.platform AS platform
      COMMENT = '配信プラットフォーム',
    raw_ad_creatives.start_date AS start_date
      COMMENT = '配信開始日',
    raw_ad_creatives.end_date AS end_date
      COMMENT = '配信終了日',
    raw_ad_creatives.impressions AS impressions
      COMMENT = 'インプレッション数',
    raw_ad_creatives.clicks AS clicks
      COMMENT = 'クリック数',
    raw_ad_creatives.conversions AS conversions
      COMMENT = 'コンバージョン数',
    raw_ad_creatives.spend AS spend
      COMMENT = '広告費用（円）',
    gold_ad_image_analysis.has_person AS has_person
      COMMENT = '人物が写っているか（true / false）',
    gold_ad_image_analysis.has_mascot AS has_mascot
      COMMENT = 'マスコットキャラクターが登場するか（true / false）',
    gold_ad_image_analysis.color_tone AS color_tone
      COMMENT = '色調（暖色系 / 寒色系）',
    gold_ad_image_analysis.is_complex AS is_complex
      COMMENT = '情報量が多い複雑なレイアウトか（true / false）',
    gold_ad_image_analysis.has_number_highlight AS has_number_highlight
      COMMENT = '数字（% / 金利 / 円）が強調表示されているか（true / false）'
  )
  METRICS (
    raw_ad_creatives.acp_score AS DIV0(SUM(conversions), SUM(impressions)) * 10000
      COMMENT = 'ACPスコア = CTR x CVR x 10000（社内 KPI）'
  );

## 7. 再チャレンジ！（改善後の View に付け替え）

新しい Semantic View を作成しました。エージェントのツールを付け替えて、もう一度同じ質問をしてみましょう。

### View の付け替え手順

#### Step 1: 旧ツールを削除する

1. Snowsight の左メニューから **「AI & ML」→「エージェント」** をクリック
2. `Ad_Performance_Navi` を選択して開く
3. 「ツール」タブを開き、`ad_performance_analysis`（旧 View）の **「…」→「削除」** をクリック

#### Step 2: 新しい Semantic View を追加する

1. **「Cortex アナリスト」** の「＋追加」をクリック
2. **「セマンティックビューを追加」** をクリック
3. `GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.SV_AD_PERFORMANCE_ENHANCED` を選択
4. 名前を入力（例: `ad_performance_enhanced`）
5. 説明を入力（例: `ACPスコアと画像特徴量を含む広告分析`）
6. ウェアハウスに `GLACIERSTYLE_WH` を指定
7. **「保存」** をクリック

#### Step 3: CoWork で再度質問する

1. Snowsight の左メニューから **「AI & ML」→「Snowflake Intelligence」** をクリック
2. `Ad_Performance_Navi` を選択
3. チャット欄に質問を入力

### 再度試してみる質問

```
ACPスコアが高い広告クリエイティブの傾向を教えてください
```

```
人物が写っている広告とそうでない広告で、どんな違いがありますか？
```

```
暖色系と寒色系のバナーでクリック率に差はありますか？
```

> **確認ポイント:**
> - 先ほど答えられなかった質問に、今度は正しく回答できるか確認しましょう
> - CoWork が画像特徴量と配信実績を **JOIN した SQL** を自動生成していることを確認しましょう
